#Set-up Environment

In [3]:
%pip install fastf1

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install pandas matplotlib seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install scipy statsmodels

Note: you may need to restart the kernel to use updated packages.


In [7]:
#Data Extraction & Environment
import os
import fastf1
import pandas as pd
import numpy as np

#Anomaly Detection
from scipy import stats # For Z-score calculations

# EDA / (ANOVA & Mixed-Effects)
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf # <-- mixed Effects models

#Additive Predictive Modeling
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score
from scipy.optimize import curve_fit # for the logarithmic track evo

#Dynamic Programming & Optimization
from functools import lru_cache # built-in memoization


# 1 Data Acquisition

In [ ]:
cache_dir = 'f1_cache'
os.makedirs(cache_dir, exist_ok=True)
fastf1.Cache.enable_cache(cache_dir)

#---

print("----2024 BAHRAIN FP2 SESSION (For the compound delta)----")
session_fp2 = fastf1.get_session(2024, 'Bahrain', 'FP2')
session_fp2.load()

#----


----2024 BAHRAIN FP2 SESSION (For the compound delta)----


core           INFO 	Loading data for Bahrain Grand Prix - Practice 2 [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 

In [10]:
print("----2024 BAHRAIN ACTUAL RACE SESSION (For models and dp)----")
session_race = fastf1.get_session(2024, 'Bahrain', 'R')
session_race.load()

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


----2024 BAHRAIN ACTUAL RACE SESSION (For models and dp)----


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

In [11]:
#REference driver
reference_driver = 'VER' #Max Verstappen
laps_race = session_race.laps
ver_laps = laps_race.pick_driver(reference_driver)

print(f"\nLoaded {len(ver_laps)} race laps for {reference_driver}.")


Loaded 57 race laps for VER.


d:\PersonalProjects\F1-Pitstop-Optimization-DynamicProgamming-AdditiveModeling\.conda\Lib\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"


In [16]:
laps_fp2 = session_fp2.laps
laps_race = session_race.laps

ver_laps_race = laps_race.pick_driver('VER')
essential_columns = ['LapNumber', 'LapTime', 'Compound', 'TyreLife', 'Stint', 'TrackStatus']

print("---2024 Bahrain Race Data: Max Verstappen---")
display(ver_laps_race[essential_columns].head(31))


print("\nAll Available Columns in the Dataset")
print(ver_laps_race.columns.tolist())

---2024 Bahrain Race Data: Max Verstappen---


d:\PersonalProjects\F1-Pitstop-Optimization-DynamicProgamming-AdditiveModeling\.conda\Lib\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"


,LapNumber,LapTime,Compound,TyreLife,Stint,TrackStatus
0,1.0,0 days 00:01:37.284000,SOFT,4.0,1.0,12
1,2.0,0 days 00:01:36.296000,SOFT,5.0,1.0,1
2,3.0,0 days 00:01:36.753000,SOFT,6.0,1.0,1
3,4.0,0 days 00:01:36.647000,SOFT,7.0,1.0,1
4,5.0,0 days 00:01:37.173000,SOFT,8.0,1.0,1
5,6.0,0 days 00:01:37.092000,SOFT,9.0,1.0,1
6,7.0,0 days 00:01:37.038000,SOFT,10.0,1.0,1
7,8.0,0 days 00:01:37.024000,SOFT,11.0,1.0,1
8,9.0,0 days 00:01:37.229000,SOFT,12.0,1.0,1
9,10.0,0 days 00:01:36.960000,SOFT,13.0,1.0,12



All Available Columns in the Dataset
['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate']


# Data Preprocessing (ETL method)